On s'occupe des tables des constructeurs

In [13]:
import numpy as np
import pandas as pd
#importation des données
cst = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructors.csv")
cst_res = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructor_results.csv")
cst_stand = pd.read_csv("/Users/gabriels./Desktop/ENSAI /programmation /gab_projet_TDD/donnees_formule_un/constructor_standings.csv")
race = pd.read_csv("/Users/gabriels./Desktop/projet_info2/donnees_formule_un/races.csv")
#cst_stand = pd.read_csv("donnees_formule_un/constructor_standings.csv")
#st_res = pd.read_csv("donnees_formule_un/constructor_results.csv")

structure de la table cst :
- constructorID = int
- constructorRef = object
- name = object
- nationality = object
- url = object

structure de la table cst_res:
- constructorResultsID = int
- raceID = int 
- constructorID = int
- points = float
- status = object

structure de la table cst_stand:
- constructorStandingsID = int
- raceID = int
- constructorID = int
- points = float
- positionText = object
- wins = int

In [25]:
#recherche des Na : on a des \\N = équivalent
cst
cst[cst.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

cst_stand
cst_stand[cst_stand.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
#pas de \\N

cst_res
cst_res[cst_res.apply(lambda row: row.astype(str).str.contains(r"\\N", na=False).any(), axis=1)]
# un \\N à chaque ligne


#on remplace les \N par des NA:
cst_res.replace("\\N", np.nan, inplace= True)


race.replace("\\N", np.nan, inplace= True)

Question : quelle écurie a gagné le plus de courses ? 
Faire un classement des écuries selon le nombre de victoires cumulées

In [139]:
#on fait un groubpy pour récupérer le constructorId ayant remporté le plus de points pour chaque course 
max_course = cst_res.loc[:,['constructorId', 'raceId', 'points']].groupby('raceId').agg(premier = ('points', "max"))
#la colonne constructorId n'est pas affiché : on va merge avec d'autres tables pour récupérer les id puis les noms des constructeurs
max_course.reset_index()
max_course = pd.merge(max_course, race,on = "raceId" ,how = "inner")
max_course = pd.merge(max_course, cst_res, on = "raceId", how = "inner")
max_course = max_course.loc[:, ["constructorId", "points", "name", "date"]]
max_course = pd.merge(max_course, cst, on = "constructorId", how = "inner")
max_course = max_course.rename(columns = {'name_x' : 'course'})
max_course = max_course.rename(columns={'name_y' : 'constructor'}) #on a une table qui recense les informations qu'on souhaite

#on peut faire le groupby:
max_course
classement = max_course.groupby("constructor").size() #compte le nombre d'occurence pour chaque constructeur
classement.sort_values(ascending = False)




constructor
Ferrari        1032
McLaren         917
Williams        831
Tyrrell         433
Sauber          407
               ... 
Protos            1
Fry               1
Connew            1
RE                1
Cooper-OSCA       1
Length: 175, dtype: int64

La table ci-dessus indique que le contsructeur qui a cumulé le plus de courses remportées d'après la table race est ferrari

Question 2: quel constructeur a remporté le plus de saisons ? 

In [152]:
#on veut récupérer la course la plus ancienne de la table race
race["date"].min()
#la première course de la base de donnée a été effectuée le 13/05/1950
season = pd.read_csv("/Users/gabriels./Desktop/projet_info2/donnees_formule_un/seasons.csv")
season #on remarque que les saisons sont découpées par années et qu'il n'y a pas de "débordement" d'une année à l'autre
#l'écurie qui aura remporté le plus de points dans la saison la remporte 
#on veut regrouper à la fois par année et par constructeur 

total_course = pd.merge(race, cst_res, how = "inner")
total_course = total_course.loc[:, ["raceId", "year", "constructorId", "points"]]
total_course = pd.merge(total_course, cst, how = "right")
total_course.groupby(["year", "constructor"]).agg(points = ('points', 'max'))

KeyError: 'constructor'